# Reinforcement learning i et lite rutenett

**Du skal kunne:** skille tilstand, handling, belønning og policy; forklare
utforsking og oppdatering av en Q-verdi. Vi bruker klassisk tabellbasert Q-learning.
Det er enklere å se læringsregelen her før man kombinerer den med nevrale nett.
GPU gir ingen nødvendig fordel for denne lille tabellen.

Agenten starter øverst til venstre i et 4 × 4-rutenett. Den skal finne målet
nederst til høyre og unngå hull. Det ligner klassiske gridworld- og FrozenLake-oppgaver,
men her er miljøet deterministisk og skrevet ut i full lengde.


## Gjør arbeidsbenken klar

`import` henter verktøy; `as` gir et kortere navn. `Path` håndterer filstier.
Vi finner kursmappen og legger `startkode` i Pythons søkesti, slik at samme kode
virker fra VS Code og JupyterLab. `sett_seed(42)` gir et fast startpunkt for
slumptall; det gjør sammenligning lettere, men lover ikke identiske tall på ulike
maskiner. `velg_enhet()` velger NVIDIA-GPU når CUDA er tilgjengelig, ellers CPU.
En **enhet** er stedet beregningene skjer. Modell og data må ligge på samme enhet.

**Før kjøring:** Gjett om utskriften blir `cpu` eller `cuda`.
**Etter kjøring:** Kontroller at det stemmer med installasjonssjekken.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

# Finn kursmappen både fra en notatbok og fra prosjektroten.
her = Path.cwd().resolve()
kandidater = [her, *her.parents, her / "05-dypere-maskinlaering"]
kursmappe = next(p for p in kandidater if (p / "startkode" / "kursverktoy.py").exists())
if str(kursmappe / "startkode") not in sys.path:
    sys.path.insert(0, str(kursmappe / "startkode"))
from kursverktoy import sett_seed, velg_enhet
sett_seed(42)
enhet = velg_enhet()
print("PyTorch:", torch.__version__, "| beregner på:", enhet)


## Skriv reglene for verden

En **tilstand** er nummeret på ruten (0–15), en **handling** er en av fire retninger.
`divmod` gir rad og kolonne. Ved kanten blir agenten stående. Målet gir +1, hull
−1, og andre steg −0,01 slik at unødvendige omveier har en kostnad.
`ferdig` markerer en terminal tilstand: episoden er over. En **episode** er ett
forsøk fra start til mål eller hull; vi avbryter også etter maks 60 steg.

Vi har skrevet miljøet, men læringsagenten får bare svar fra `flytt` etter hver
handling. Den får ikke en ferdig liste med riktige bevegelser.


In [ ]:
import numpy as np
hull = {5, 7, 11, 12}
maalrute = 15
retninger = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # opp, høyre, ned, venstre

def flytt(tilstand, handling):
    rad, kol = divmod(tilstand, 4)
    dr, dk = retninger[handling]
    ny_rad, ny_kol = max(0, min(3, rad + dr)), max(0, min(3, kol + dk))
    ny = 4 * ny_rad + ny_kol
    if ny == maalrute:
        return ny, 1.0, True
    if ny in hull:
        return ny, -1.0, True
    return ny, -0.01, False

kart = np.zeros((4, 4))
for rute in hull:
    kart.flat[rute] = -1
kart.flat[maalrute] = 1
plt.imshow(kart, cmap="RdYlGn", vmin=-1, vmax=1)
for rute in range(16):
    rad, kol = divmod(rute, 4)
    plt.text(kol, rad, "Mål" if rute == 15 else "Hull" if rute in hull else str(rute),
             ha="center", va="center")
plt.title("Start i rute 0")
plt.show()


## Fra erfaring til Q-verdi

`Q[s, a]` anslår framtidig samlet, diskontert belønning ved å velge handling a i
rute s og deretter handle godt. Tenk på en notisbok som får bedre rutevurderinger
etter hvert forsøk. **Policy** er regelen for å velge handling.

Vi bruker ε-greedy: med sannsynlighet ε utforsker vi tilfeldig, ellers velger vi
høyest Q-verdi. `beste_handling` trekker tilfeldig mellom like gode valg, slik at
starttabellens nuller ikke alltid betyr «gå opp». `gamma=0.95` gjør senere belønning
litt mindre verdt. `alpha=0.1` er oppdateringsraten.

Oppdateringen er `Q[s,a] += alpha * (mål - Q[s,a])`, der målet er belønning pluss
`gamma * max(Q[neste])`. I en terminal tilstand finnes ingen videre belønning;
da bruker vi bare belønningen. Dette er en læringsregel, ikke backpropagation.


In [ ]:
rng = np.random.default_rng(42)
Q = np.zeros((16, 4))
alpha, gamma = 0.1, 0.95

def beste_handling(q_rad, generator):
    like_beste = np.flatnonzero(q_rad == q_rad.max())
    return int(generator.choice(like_beste))

belonninger = []
for episode in range(2000):
    tilstand, samlet = 0, 0.0
    epsilon = max(0.05, 1.0 - episode / 1500)
    for steg in range(60):
        handling = int(rng.integers(4)) if rng.random() < epsilon else beste_handling(Q[tilstand], rng)
        neste, belonning, ferdig = flytt(tilstand, handling)
        framtid = 0.0 if ferdig else gamma * Q[neste].max()
        maalverdi = belonning + framtid
        Q[tilstand, handling] += alpha * (maalverdi - Q[tilstand, handling])
        samlet += belonning
        tilstand = neste
        if ferdig:
            break
    belonninger.append(samlet)
print("Q fra start:", Q[0])


## Mål uten videre læring

Treningsbelønningene inkluderer bevisste utforskingsfeil. Derfor måler vi også en
fast policy etterpå med ε = 0, og sammenligner med tilfeldig handling. Vi oppdaterer
ikke Q i evalueringen. Den separate generatoren gir et gjentakbart måleforsøk.
I denne deterministiske verdenen kan en god policy klare alle forsøk; det sier lite
om evnen til å håndtere et ukjent kart eller tilfeldig glidning.

`convolve` med 100 like vekter lager et glidende gjennomsnitt av belønningene.
Det jevner ut tilfeldige svingninger, men skjuler noe av variasjonen.


In [ ]:
def evaluer(tilfeldig=False, antall=200):
    generator = np.random.default_rng(123)
    seire = 0
    for _ in range(antall):
        tilstand = 0
        for steg in range(60):
            handling = int(generator.integers(4)) if tilfeldig else beste_handling(Q[tilstand], generator)
            tilstand, belonning, ferdig = flytt(tilstand, handling)
            if ferdig:
                seire += int(tilstand == maalrute)
                break
    return seire / antall

print(f"Tilfeldig policy: {evaluer(True):.1%} når målet")
print(f"Lært policy: {evaluer(False):.1%} når målet")
plt.plot(np.arange(100, len(belonninger)+1), np.convolve(belonninger, np.ones(100)/100, mode="valid"))
plt.xlabel("Episode")
plt.ylabel("Gjennomsnittlig belønning siste 100")
plt.show()


## Se policyen og følg en rute

Pilene viser en av de beste handlingene per rute. Her bruker `argmax` første beste
ved likhet; under trening og evaluering brøt vi likheter tilfeldig. Ubesøkte ruter
kan derfor få en vilkårlig pil. Vi viser også faktisk rute fra start og stopper
ved mål, hull eller 20 steg så en løkke ikke fortsetter uendelig.


In [ ]:
piler = ["↑", "→", "↓", "←"]
for rad in range(4):
    print(" ".join("M" if 4*rad+kol == maalrute else "X" if 4*rad+kol in hull
                   else piler[int(Q[4*rad+kol].argmax())] for kol in range(4)))
tilstand, rute = 0, [0]
for _ in range(20):
    tilstand, _, ferdig = flytt(tilstand, int(Q[tilstand].argmax()))
    rute.append(tilstand)
    if ferdig:
        break
print("Rute:", rute)


## Eksperiment og bro til dyp reinforcement learning

1. Sett ε til 0 fra starten, tren fra en ny nulltabell og sammenlign flere seeds.
   Hvorfor kan begrenset utforsking gjøre læringen dårligere?
2. Endre stegbelønningen til +0,1. Undersøk om agenten foretrekker omveier.
   Det viser at den optimaliserer belønningen vi skrev, som kan avvike fra hensikten.
3. Flytt et hull **etter** trening. Hvorfor må policyen vurderes på nytt?

I dyp Q-learning erstatter et nevralt nett tabellen. Et vanlig DQN-oppsett bruker
også replay-buffer og et målnettverk for å stabilisere læringen. Andre metoder,
som policy gradient, lærer en policy mer direkte. RLHF kombinerer igjen flere
trinn og ofte menneskelige preferanser; denne oppgaven implementerer ikke RLHF.

Ved maks 60 steg avbryter vi forsøket av praktiske grunner. Det er en tidsgrense,
ikke et hull eller mål, så Q-målet i siste steg beholder framtidsleddet.

**Kontroll:** Hva er forskjellen på fasit i bildeoppgaven og belønning her?
Bildeoppgaven får riktig klasse for hvert bilde. Her får vi konsekvensen av en
handling og må lære hvilke handlinger som gir god samlet belønning.

Kilde: [Watkins og Dayan om Q-learning](https://www.gatsby.ucl.ac.uk/~dayan/papers/cjch.pdf).
**Avslutning:** Velg et spørsmål i [prosjektoppgaven](../PROSJEKT.md).
